# Bayesian GARCH(1,1) — Random Walk Metropolis Sampler

## Notations

---

### Reference  
Following the model and priors described in:  
Mira, Solgi & Imparato (2013) *"Zero variance Markov chain Monte Carlo for Bayesian estimators"*, Statistics and Computing 23:653–662.

---

## Model

\[
y_t = \sqrt{h_t} \cdot \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0,1) \; (\text{or Student-}t)
\]

\[
h_t = \omega + \alpha \, y_{t-1}^2 + \beta \, h_{t-1}
\]

---

## Parameters

\[
\theta = (\omega, \alpha, \beta)
\]

### Constraints

- \(\omega > 0\)  
- \(\alpha \geq 0\)  
- \(\beta \geq 0\)  
- \(\alpha + \beta < 1\)

---

## Prior Distribution

(Ardia 2008 / Nakatsuma 2000 setup)

- \(p(\omega) \propto \frac{1}{\omega}\)  *(log-uniform / Jeffreys-type prior)*  
- \(\alpha \sim \mathcal{U}(0,1)\)  
- \(\beta \sim \mathcal{U}(0,1)\)  
- Subject to: \(\alpha + \beta < 1\)

---

## Log-Posterior (up to a constant)

\[
\log \pi(\theta \mid y) = \log L(\theta; y) + \log p(\theta)
\]

---

## Random Walk Metropolis Algorithm

### Proposal

\[
\theta^* = \theta + \delta, \quad \delta \sim \mathcal{N}(0, \Sigma)
\]

- \(\Sigma\) is tuned to achieve an acceptance rate of approximately **23%**  
  (rule of thumb from Gelman et al.)

---

### Practical Implementation

- Perform sampling in a **transformed (unconstrained) space**:
  - \(\log \omega\)
  - \(\text{logit}(\alpha)\)
  - \(\text{logit}(\beta)\)

- Then map back to the original parameter space to enforce constraints.

---

In [ ]:
!pip install -r requirements.txt

# 1.  GARCH(1,1) utilities

In [1]:
# ─────────────────────────────────────────────────────────────
#  1.  GARCH(1,1) utilities
# ─────────────────────────────────────────────────────────────

def garch_variances(y, omega, alpha, beta):
    """Recursively compute conditional variances h_t."""
    T = len(y)
    h = np.empty(T)
    h[0] = np.var(y)           # initialise at sample variance
    for t in range(1, T):
        h[t] = omega + alpha * y[t-1]**2 + beta * h[t-1]
    return h


def log_likelihood(y, omega, alpha, beta):
    """Gaussian GARCH(1,1) log-likelihood (sum over t=1..T)."""
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return -np.inf
    h = garch_variances(y, omega, alpha, beta)
    # avoid numerical issues
    if np.any(h <= 0):
        return -np.inf
    return -0.5 * np.sum(np.log(h) + y**2 / h)


def log_prior(omega, alpha, beta):
    """Log-prior: log-uniform on omega, Uniform on alpha & beta,
    with the stationarity constraint alpha+beta < 1."""
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return -np.inf
    # log p(omega) ∝ -log(omega)   (Jeffreys prior)
    return -np.log(omega)


def log_posterior(y, omega, alpha, beta):
    lp = log_prior(omega, alpha, beta)
    if not np.isfinite(lp):
        return -np.inf
    return log_likelihood(y, omega, alpha, beta) + lp

# Parameter transformations  (constrained → unconstrained)

In [2]:
# ─────────────────────────────────────────────────────────────
#  2.  Parameter transformations  (constrained → unconstrained)
# ─────────────────────────────────────────────────────────────
# theta = (omega, alpha, beta)   constrained
# phi   = (log_omega, logit_alpha, logit_beta_given_alpha)  unconstrained
#
# alpha  = sigmoid(phi[1])
# beta   = (1-alpha) * sigmoid(phi[2])   ensures alpha+beta < 1

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def logit(p):
    return np.log(p / (1.0 - p))

def phi_to_theta(phi):
    """Unconstrained → constrained."""
    log_omega, phi_alpha, phi_beta = phi
    omega = np.exp(log_omega)
    alpha = sigmoid(phi_alpha)
    beta  = (1.0 - alpha) * sigmoid(phi_beta)
    return omega, alpha, beta

def theta_to_phi(omega, alpha, beta):
    """Constrained → unconstrained."""
    log_omega = np.log(omega)
    phi_alpha = logit(alpha)
    phi_beta  = logit(beta / (1.0 - alpha))
    return np.array([log_omega, phi_alpha, phi_beta])

def log_jacobian(phi):
    """
    Log |∂theta/∂phi| for the change of variables.
    Needed so the RWM in phi-space targets the right posterior.
    """
    _, phi_alpha, phi_beta = phi
    alpha = sigmoid(phi_alpha)
    s_b   = sigmoid(phi_beta)
    # d(omega)/d(log_omega) = omega
    # d(alpha)/d(phi_alpha) = alpha*(1-alpha)
    # d(beta)/d(phi_beta)   = (1-alpha)*s_b*(1-s_b)
    log_J = (phi[0]                          # log omega
             + np.log(alpha*(1-alpha))        # dalpha/dphi_alpha
             + np.log((1-alpha)*s_b*(1-s_b))) # dbeta/dphi_beta
    return log_J

def log_posterior_transformed(y, phi):
    """Log-posterior in unconstrained space (includes Jacobian)."""
    omega, alpha, beta = phi_to_theta(phi)
    lp = log_posterior(y, omega, alpha, beta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_jacobian(phi)

# 3.  Random Walk Metropolis sampler

In [3]:
# ─────────────────────────────────────────────────────────────
#  3.  Random Walk Metropolis sampler
# ─────────────────────────────────────────────────────────────

def rwm_garch(y, n_iter=50_000, burn_in=10_000, sigma_prop=None,
              seed=42, verbose=True):
    """
    Random Walk Metropolis for the GARCH(1,1) posterior.
    Sampling is done in the unconstrained (phi) space.

    Parameters
    ----------
    y         : array-like, log-returns
    n_iter    : total MCMC iterations (including burn-in)
    burn_in   : number of draws to discard
    sigma_prop: proposal standard deviations (len-3 array).
                If None, auto-tuned via a short pilot run.
    seed      : random seed
    verbose   : print acceptance rate and progress

    Returns
    -------
    samples   : dict with arrays 'omega','alpha','beta' (post burn-in)
    accept_rate: float
    """
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=float)

    # ── Starting values: rough MLE-style initialisation ──
    omega0 = np.var(y) * 0.05
    alpha0 = 0.10
    beta0  = 0.85
    phi_cur = theta_to_phi(omega0, alpha0, beta0)
    lp_cur  = log_posterior_transformed(y, phi_cur)

    # ── Proposal covariance (diagonal) ──
    if sigma_prop is None:
        # Pilot run of 2000 iterations with wide steps to estimate scale
        sigma_prop = np.array([0.05, 0.10, 0.10])
        if verbose:
            print("Running pilot to calibrate proposal ...")
        sigma_prop = _calibrate_proposal(y, phi_cur, sigma_prop,
                                         rng, n_pilot=3000)
        if verbose:
            print(f"  Calibrated σ_prop = {sigma_prop.round(4)}")

    # ── Main chain ──
    chain = np.empty((n_iter, 3))
    n_accept = 0

    for i in range(n_iter):
        # Proposal
        phi_prop = phi_cur + sigma_prop * rng.standard_normal(3)
        lp_prop  = log_posterior_transformed(y, phi_prop)

        # Metropolis ratio (symmetric proposal → only posterior ratio)
        log_alpha_mh = lp_prop - lp_cur
        if np.log(rng.uniform()) < log_alpha_mh:
            phi_cur = phi_prop
            lp_cur  = lp_prop
            n_accept += 1

        chain[i] = phi_to_theta(phi_cur)

        if verbose and (i+1) % 10_000 == 0:
            ar = n_accept / (i+1)
            print(f"  Iter {i+1:>6d}  accept = {ar:.3f}")

    accept_rate = n_accept / n_iter
    samples = {
        'omega': chain[burn_in:, 0],
        'alpha': chain[burn_in:, 1],
        'beta' : chain[burn_in:, 2],
    }
    return samples, accept_rate


def _calibrate_proposal(y, phi_start, sigma_init, rng, n_pilot=3000,
                        target_ar=0.23):
    """Simple adaptive scaling: adjust σ to hit ~23% acceptance rate."""
    sigma = sigma_init.copy()
    phi_cur = phi_start.copy()
    lp_cur  = log_posterior_transformed(y, phi_cur)
    n_accept = 0
    for i in range(n_pilot):
        phi_prop = phi_cur + sigma * rng.standard_normal(3)
        lp_prop  = log_posterior_transformed(y, phi_prop)
        if np.log(rng.uniform()) < lp_prop - lp_cur:
            phi_cur = phi_prop
            lp_cur  = lp_prop
            n_accept += 1
        # Adapt every 500 iterations
        if (i+1) % 500 == 0:
            ar = n_accept / (i+1)
            sigma *= np.exp(ar - target_ar)   # simple Robbins-Monro step
    return sigma

